# Session 3: the arteries as an elastic chamber (Windkessel)
**A. Real-world problem.** The heart does not push steadily: it ejects blood in a short
burst (systole) and rests (diastole). Yet the blood pressure in the arteries never drops to
zero between beats. The big arteries are elastic: they swell during the burst, store blood,
and squeeze it onwards while the heart rests, like the air chamber of an old fire engine
pump (German: Windkessel). What sets the top and bottom pressure, and what happens when the
arteries get stiffer?

*This is general education. It is not medical advice and the model is not a diagnostic
tool. The 120/80 target below is an illustrative number, not a norm for any person.*

**Plan.** (1) write the model, one equation for the pressure P, (2) solve it with our own
RK4, (3) validate against the exact decay in diastole and the mean-flow balance, (4) stiffen
the arteries, change the heart rate, and break the model on purpose.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

HR0, SV0 = 70.0, 70.0          # typical textbook heart rate (beats/min) and stroke volume (mL)
SYSTOLE_FRACTION = 0.3         # the ejection lasts about 0.3 of the beat
N_STEPS = 300                  # RK4 steps per beat (a multiple of 10, so systole ends on a step)

## Step 1: the model
Two parts: an elastic compliance `C` (mL per mmHg: how much blood the arteries store per
mmHg of pressure rise) and a peripheral resistance `R` (mmHg per mL/s). The ventricle
pushes in a flow `Q_in(t)`, a half sine during the first 0.3 of each beat and zero after.
The blood stored in `C` grows with inflow and drains through `R`:

    C dP/dt = Q_in(t) - P / R

The peak of the half sine is fixed so that one beat delivers exactly the stroke volume:
`SV = integral of Q_in dt = 2 Q_peak T_s / pi`.

In [ ]:
def inflow(t, hr=HR0, sv=SV0, frac=SYSTOLE_FRACTION):
    """Flow from the ventricle (mL/s) at time t (s): half sine in systole, zero in diastole."""
    period = 60.0 / hr
    ts = frac * period
    phase = t % period
    if phase >= ts:
        return 0.0
    return math.pi * sv / (2.0 * ts) * math.sin(math.pi * phase / ts)

## Step 2: our own RK4
The state is a vector `y` (here only one number, the pressure P). Runge-Kutta 4 takes four
slope estimates per step and combines them; its error shrinks like dt^4.

In [ ]:
def rk4_step(f, t, y, dt):
    k1 = f(t, y)
    k2 = f(t + dt / 2, y + dt / 2 * k1)
    k3 = f(t + dt / 2, y + dt / 2 * k2)
    k4 = f(t + dt, y + dt * k3)
    return y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def windkessel(R, C, hr=HR0, sv=SV0, beats=30, n=N_STEPS, frac=SYSTOLE_FRACTION, p0=None):
    """Integrate C dP/dt = Q_in - P/R. Returns time (s) and pressure (mmHg) arrays.

    R in mmHg s/mL, C in mL/mmHg. p0 defaults to the mean-flow pressure R * mean(Q_in).
    """
    if R <= 0 or C <= 0:
        raise ValueError("R and C must be positive")
    period = 60.0 / hr
    dt = period / n
    if p0 is None:
        p0 = R * sv * hr / 60.0
    f = lambda t, y: np.array([(inflow(t, hr, sv, frac) - y[0] / R) / C])
    y = np.array([float(p0)])
    t = np.arange(beats * n + 1) * dt
    P = np.empty(beats * n + 1)
    P[0] = y[0]
    for k in range(beats * n):
        y = rk4_step(f, t[k], y, dt)
        P[k + 1] = y[0]
    return t, P


def last_beat(R, C, hr=HR0, sv=SV0, beats=30, n=N_STEPS, frac=SYSTOLE_FRACTION):
    """Time (from 0) and pressure over the final beat, after the start-up has died away."""
    t, P = windkessel(R, C, hr, sv, beats, n, frac)
    return t[:n + 1] - t[0], P[-(n + 1):]

## Step 3: read the waveform
Systolic = highest pressure, diastolic = lowest, pulse pressure = their difference, mean =
time average over one beat (the last point repeats the first, so it is left out).

In [ ]:
def bp_stats(P):
    """(systolic, diastolic, pulse pressure, mean) in mmHg from one periodic beat."""
    body = np.asarray(P)[:-1]
    return float(body.max()), float(body.min()), float(body.max() - body.min()), float(body.mean())


def diastolic_decay(P0, R, C, t):
    """Exact solution of C dP/dt = -P/R with P(0) = P0, for t seconds after systole ends."""
    return P0 * np.exp(-np.asarray(t) / (R * C))


def mean_inflow(hr=HR0, sv=SV0):
    return sv * hr / 60.0          # mL/s

## Step 4: validate
Two independent checks. (a) In diastole there is no inflow, so the exact solution is
P = P0 exp(-t / (R C)); the numerical curve must lie on it. (b) Averaged over a periodic
beat the stored blood does not change, so mean(P) / R = mean(Q_in): mean P = R x mean Q.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    R, C = 1.214, 1.251                       # tuned in the mini-project to give about 120/80
    tt, P = last_beat(R, C)
    period = 60.0 / HR0
    ts = SYSTOLE_FRACTION * period
    s, d, pp, m = bp_stats(P)
    print(f"R = {R}, C = {C}: systolic {s:.1f}, diastolic {d:.1f}, pulse {pp:.1f}, mean {m:.1f} mmHg")
    print(f"mean P / R = {m / R:.2f} mL/s, mean inflow = {mean_inflow():.2f} mL/s")
    i0 = int(round(SYSTOLE_FRACTION * N_STEPS))
    exact = diastolic_decay(P[i0], R, C, tt[i0:] - ts)
    print(f"largest gap between RK4 and the exact diastolic decay: {np.abs(P[i0:] - exact).max():.2e} mmHg")

    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(tt, P, lw=2, label="RK4 solution")
    ax.plot(tt[i0:], exact, "k--", lw=1, label="exact decay in diastole")
    ax.axhline(m, color="gray", ls=":", label=f"mean {m:.0f} mmHg")
    ax.axvspan(0, ts, color="tab:red", alpha=0.08, label="systole (ejection)")
    ax.set_xlabel("time within the beat (s)")
    ax.set_ylabel("arterial pressure (mmHg)")
    ax.set_title("Windkessel pressure wave, R = 1.214, C = 1.251")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "windkessel_wave.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    for c_val, col in ((2.5, "tab:green"), (1.251, "tab:blue"), (0.626, "tab:red")):
        t2, P2 = last_beat(R, c_val)
        s2, d2, pp2, m2 = bp_stats(P2)
        ax2.plot(t2, P2, color=col, label=f"C = {c_val}: pulse {pp2:.0f}, mean {m2:.0f} mmHg")
        print(f"C = {c_val}: systolic {s2:.1f}, diastolic {d2:.1f}, pulse {pp2:.1f}, mean {m2:.1f}")
    ax2.set_xlabel("time within the beat (s)")
    ax2.set_ylabel("arterial pressure (mmHg)")
    ax2.set_title("Stiffer arteries (smaller C): same mean, bigger swing")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "stiffness_effect.png", dpi=120)
    print("saved figures in", OUT)

## Where the model stops
The ventricle here forces its flow whatever the pressure; a real heart ejects only while
its pressure exceeds the arterial pressure and the aortic valve closes afterwards. There is
no wave travelling along the arteries, no change of C with pressure, and one lump stands
for the whole arterial tree. Very small C gives absurd pressure swings (see the checkpoint).

## Coding tasks
1. **Easy.** Run `last_beat(1.214, c)` for c = 0.8, 1.251 and 2.0 and print the pulse pressure with `bp_stats`. Predict the order first.
2. **Stretch.** Replace `rk4_step` by a simple Euler step and compare the largest error against the exact diastolic decay for 60, 120 and 240 steps per beat. How does the error change when you double the steps?
3. **Explain.** Explain why the mean pressure of the model for 120/80 (99.1 mmHg) is higher than the rule-of-thumb value 93.3, and what this says about trusting a rule of thumb.

## Assessment
- **A. Conceptual.** Why does the arterial pressure not drop to zero in diastole when the heart is not pushing? Why does a stiffer artery change the pulse pressure but not the mean pressure?
- **B. Numerical.** With R = 1.214, a heart rate of 60 beats per minute and a stroke volume of 70 mL, find the mean pressure from the mean-flow balance. Compare with the 70 beats case (99.1 mmHg).
- **C. Graph and simulation interpretation.** In the stiffness figure the three waves have pulse pressures 20.1, 40.0 and 78.9 mmHg for C = 2.5, 1.251 and 0.626. What pattern do you see, and what is the mean pressure in each case?
- **D. Design and optimisation challenge.** Design: choose R and C so that the model has a mean pressure of 90 mmHg and a pulse pressure close to 30 mmHg at 70 beats per minute and 70 mL. Report the systolic and diastolic values. This is an illustrative exercise, not a target for any person.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    R, C = 1.214, 1.251
    n = L.N_STEPS
    period = 60.0 / L.HR0
    ts = L.SYSTOLE_FRACTION * period
    t, P = L.last_beat(R, C)
    s, d, pp, m = L.bp_stats(P)

    # --- the inflow
    dt = 1e-4
    tt = np.arange(0.0, period, dt)
    q = np.array([L.inflow(x) for x in tt])
    checks.append(("one beat delivers the stroke volume 70 mL", abs(q.sum() * dt - 70.0) < 0.01))
    checks.append(("the inflow is zero in diastole and peaks at 427.6 mL/s",
                   float(q[tt > ts + 1e-9].max()) == 0.0 and abs(q.max() - 427.6) < 0.1))
    checks.append(("mean inflow is 81.67 mL/s (4.9 L/min)", abs(L.mean_inflow() - 81.667) < 1e-3))

    # --- worked example (Section E)
    rc = R * C
    checks.append(("worked example: time constant R C = 1.519 s", abs(rc - 1.5187) < 1e-3))
    factor = np.exp(-(period - ts) / rc)
    checks.append(("worked example: diastole lasts 0.600 s and the pressure falls by a factor 0.674",
                   abs((period - ts) - 0.6) < 1e-12 and abs(factor - 0.6736) < 1e-4))
    p_ts = P[int(round(L.SYSTOLE_FRACTION * n))]
    checks.append(("worked example: pressure 119.3 at the end of ejection becomes 80.4 at the end of the beat",
                   abs(p_ts - 119.29) < 0.02 and abs(p_ts * factor - 80.36) < 0.02 and abs(P[-1] - 80.36) < 0.02))
    checks.append(("worked example: R x mean flow = 1.214 x 81.67 = 99.1 mmHg equals the mean pressure",
                   abs(R * L.mean_inflow() - 99.15) < 0.01 and abs(m - 99.14) < 0.02))

    # --- the target waveform (mini-project values)
    checks.append(("R = 1.214 and C = 1.251 give systolic 120 and diastolic 80 within 0.1 mmHg",
                   abs(s - 120.0) < 0.1 and abs(d - 80.0) < 0.1))
    checks.append(("pulse pressure is 40 mmHg for the tuned model", abs(pp - 40.0) < 0.2))
    checks.append(("the minimum comes just after the next ejection starts, so it is below the end-of-beat value by 0.3",
                   abs(P[-1] - d - 0.33) < 0.05 and int(np.argmin(P[:-1])) < 10))
    checks.append(("the untuned guess R = 1.143, C = 1.0 misses the target (diastolic below 71)",
                   L.bp_stats(L.last_beat(1.143, 1.0)[1])[1] < 71.0))

    # --- validation 1: exact decay in diastole
    i0 = int(round(L.SYSTOLE_FRACTION * n))
    exact = L.diastolic_decay(P[i0], R, C, t[i0:] - ts)
    checks.append(("RK4 matches the exact diastolic decay to 1e-6 mmHg", float(np.abs(P[i0:] - exact).max()) < 1e-6))

    # --- validation 2: mean-flow balance for several (R, C, HR)
    ok = True
    for r_, c_, h_ in ((0.8, 0.5, 70), (1.6, 2.0, 70), (1.214, 1.251, 100)):
        _, Pb = L.last_beat(r_, c_, hr=h_)
        ok &= abs(L.bp_stats(Pb)[3] - r_ * L.mean_inflow(h_)) < 0.05
    checks.append(("mean P = R x mean Q for three different (R, C, heart rate) settings", bool(ok)))

    # --- order of the method: halving dt cuts the error by about 16
    _, Pr = L.last_beat(R, C, n=1200)
    errs = [float(np.abs(L.last_beat(R, C, n=k)[1] - Pr[::1200 // k]).max()) for k in (60, 120)]
    checks.append(("RK4 is fourth order: halving the step cuts the error by about 16",
                   12.0 < errs[0] / errs[1] < 20.0))

    # --- stiffer arteries
    stats = {c_: L.bp_stats(L.last_beat(R, c_)[1]) for c_ in (2.5, 1.251, 0.626)}
    checks.append(("stiffer arteries (smaller C) give a larger pulse pressure: 20.1, 40.0, 78.9 mmHg",
                   abs(stats[2.5][2] - 20.1) < 0.2 and abs(stats[1.251][2] - 40.0) < 0.2 and abs(stats[0.626][2] - 78.9) < 0.3))
    checks.append(("the mean pressure stays 99.1 mmHg whatever C is",
                   all(abs(v[3] - 99.14) < 0.03 for v in stats.values())))
    checks.append(("halving C gives 142.1 / 63.2 mmHg", abs(stats[0.626][0] - 142.1) < 0.2 and abs(stats[0.626][1] - 63.2) < 0.2))

    # --- heart rate and sensitivity
    h60 = L.bp_stats(L.last_beat(R, C, hr=60)[1])
    h90 = L.bp_stats(L.last_beat(R, C, hr=90)[1])
    checks.append(("heart rate 60 gives 106/66 (mean 85.0); 90 gives 148/108 (mean 127.5) at fixed stroke volume",
                   abs(h60[0] - 106.0) < 0.3 and abs(h60[1] - 66.0) < 0.3 and abs(h60[3] - 85.0) < 0.1
                   and abs(h90[0] - 148.2) < 0.3 and abs(h90[1] - 108.1) < 0.3 and abs(h90[3] - 127.5) < 0.1))
    base = L.bp_stats(L.last_beat(R, C)[1])
    rp = L.bp_stats(L.last_beat(R * 1.1, C)[1])
    cp = L.bp_stats(L.last_beat(R, C * 1.1)[1])
    sp = L.bp_stats(L.last_beat(R, C, sv=77.0)[1])
    checks.append(("sensitivity: R +10% raises the mean 10% (109.1) and leaves the pulse near 40",
                   abs(rp[3] / base[3] - 1.1) < 1e-3 and abs(rp[2] - 40.0) < 0.2))
    checks.append(("sensitivity: C +10% lowers the pulse pressure to 36.4 and leaves the mean",
                   abs(cp[2] - 36.4) < 0.1 and abs(cp[3] - base[3]) < 0.02))
    checks.append(("sensitivity: stroke volume +10% raises the pulse pressure to 44.0 and the mean to 109.1",
                   abs(sp[2] - 44.0) < 0.2 and abs(sp[3] - 109.06) < 0.1))

    # --- failure case and edge cases
    tiny = L.bp_stats(L.last_beat(R, 0.05)[1])
    checks.append(("failure case: C = 0.05 gives an unphysical pulse pressure of 430 mmHg and a fall to 0",
                   abs(tiny[2] - 430.0) < 1.0 and tiny[1] < 0.1))
    checks.append(("failure case: the mean is still 99.1, so the mean alone cannot reveal the failure",
                   abs(tiny[3] - 99.14) < 0.05))
    bad = 0
    for args in ((0.0, 1.0), (1.0, 0.0), (-1.0, 1.0)):
        try:
            L.windkessel(*args)
        except ValueError:
            bad += 1
    checks.append(("non-positive R or C is rejected", bad == 3))
    checks.append(("with no inflow (sv = 0) the pressure follows the exact decay 100 exp(-t/(R C))",
                   abs(float(L.windkessel(R, C, sv=0.0, beats=2, p0=100.0)[1][-1]) - 100.0 * np.exp(-2 * period / (R * C))) < 1e-6))

    # --- numbers used in the text
    checks.append(("worked example: beat 0.857 s, systole 0.257 s, diastole 0.600 s",
                   abs(period - 0.8571) < 1e-4 and abs(ts - 0.2571) < 1e-4))
    checks.append(("halving C multiplies the pulse pressure by 1.97", abs(stats[0.626][2] / stats[1.251][2] - 1.973) < 0.01))
    checks.append(("rule of thumb for 120/80 is 93.3 but the model's own mean is 99.1", abs(80 + 40 / 3 - 93.33) < 0.01 and abs(m - 99.14) < 0.02))
    r90 = 90.0 / L.mean_inflow()
    d90 = L.bp_stats(L.last_beat(r90, 1.7)[1])
    checks.append(("design: R = 1.102 and C = 1.7 give 105.3 / 75.8, pulse 29.5, mean 90.0",
                   abs(r90 - 1.102) < 1e-3 and abs(d90[0] - 105.27) < 0.1 and abs(d90[1] - 75.78) < 0.1 and abs(d90[2] - 29.49) < 0.1 and abs(d90[3] - 90.0) < 0.05))
    checks.append(("mean P at 60 beats/min is 85.0 from the balance 1.214 x 70", abs(1.214 * L.mean_inflow(60.0) - 84.98) < 0.01))
    checks.append(("in the first 0.02 s of ejection the two-element model's pressure falls by 0.2 mmHg", abs(P[7] - P[0] + 0.224) < 0.01 and abs(t[7] - 0.02) < 1e-9))
    pa = L.bp_stats(L.last_beat(1.592, C)[1])[2]
    pb = L.bp_stats(L.last_beat(1.102, C)[1])[2]
    checks.append(("a constant C gives the same pulse pressure at mean 130 and mean 90 (ratio 1.00)", abs(pa / pb - 1.0) < 0.01))
    hi = L.bp_stats(L.last_beat(R, C, sv=84.0)[1])[3]
    checks.append(("stroke volume +20% moves the mean by about 20 mmHg", abs(hi - base[3] - 19.8) < 0.3))
    pe = 80.0
    for _ in range(1000):
        pe += 0.001 * (0.0 - pe / 1.2) / 1.25
    checks.append(("snippet: Euler gives 41.1 and the exact decay gives 41.1", round(pe, 1) == 41.1 and round(80 * np.exp(-1.0 / 1.5), 1) == 41.1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
